# 3. 2D floor planning with rotation: cover as much floor area as possible

Lecture: *Week 11 - Floor Planning*, a variant of the minimise-area model in notebook 1, with a fixed floor as on the *Amending the constraints* slide after *Clustering of components*.

**The problem.** The six items of notebook 1 (total area 124) must go on a 10 × 10 floor (area 100), so they cannot all fit. Each item may be placed (u = 1) or left out (u = 0), and may be rotated. Choose and place the items to cover as much of the floor as possible.

**How to run it.** Run the cells in order, starting with the first (it installs Pyomo and HiGHS). The solve takes a few seconds.

**Expected result.** `Objective: 100.0`: the whole floor is covered, with item 3 (6 × 4) left out. The positions and rotations can differ between runs and solvers; the objective is the same.

*Python version of the MATLAB file `5 Source Files/MATLAB/floor_planning_rotation_2d_max_item_area.m`, written for this repository: same data and constraints, with a smaller big-M (see notebook 2).*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

## Load the libraries

Pyomo builds and solves the model, NumPy holds the data and Matplotlib draws the layout.

In [ ]:
from pyomo.environ import *
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## The data

The six items of notebook 1 and a 10 × 10 floor. `M` is the smallest big-M that is still large enough (notebook 2 explains it).

In [ ]:
# Data (as in the MATLAB file: the six items from the lecture)
w = np.array([4,3,6,7,3,2])
h = np.array([5,7,4,6,3,4])
W, H = 10, 10    # floor size
# Big-M: the MATLAB file uses M = 9999; a tight value is numerically safer with HiGHS (see notebook 2).
M = W + H + max(w.max(), h.max())
no_items = len(w)

## The model

As in notebook 2: corners `x`, `y`, the pair binaries `xij`, `yij`, rotation `r` and placement `u`.

In [ ]:
# Model
model = ConcreteModel()
model.I = RangeSet(no_items)
model.J = RangeSet(no_items)
model.w = Param(model.I, initialize=lambda model, i: int(w[i-1]))
model.h = Param(model.I, initialize=lambda model, i: int(h[i-1]))

model.x = Var(model.I, within=NonNegativeReals)      # bottom-left corner
model.y = Var(model.I, within=NonNegativeReals)
model.xij = Var(model.I, model.J, within=Binary)     # which side of each other two items are
model.yij = Var(model.I, model.J, within=Binary)
model.r = Var(model.I, within=Binary)                # 1 = item rotated 90 degrees
model.u = Var(model.I, within=Binary)                # 1 = item placed on the floor

## Objective

Maximise the area placed, $\sum_i u_i\, w_i h_i$: each placed item counts its area instead of 1.

In [ ]:
# Objective: the total area of the items placed
model.obj = Objective(expr=sum(model.u[i] * int(w[i-1] * h[i-1]) for i in model.I), sense=maximize)

## Constraints

The same as notebook 2: the four non-overlap constraints, switched off by $M(2 - u_i - u_j)$ when either item is left out, and the floor constraints, released by $M(1 - u_i)$ for a left-out item.

In [ ]:
# Constraints: two placed items must not overlap (switched off by M if either item is left out)
def width(model, i):   # width of item i after rotation
    return (1 - model.r[i]) * model.w[i] + model.r[i] * model.h[i]

def height(model, i):  # height of item i after rotation
    return (1 - model.r[i]) * model.h[i] + model.r[i] * model.w[i]

# (xij, yij) = (0, 0): item i is LEFT of item j
def overlap_xij_rule(model, i, j):
    if i < j:
        return model.x[i] + width(model, i) <= model.x[j] + M*(model.xij[i,j] + model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_xij = Constraint(model.I, model.J, rule=overlap_xij_rule)

# (xij, yij) = (1, 0): item j is LEFT of item i
def overlap_xji_rule(model, i, j):
    if i < j:
        return model.x[j] + width(model, j) <= model.x[i] + M*(1 - model.xij[i,j] + model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_xji = Constraint(model.I, model.J, rule=overlap_xji_rule)

# (xij, yij) = (0, 1): item i is BELOW item j
def overlap_yij_rule(model, i, j):
    if i < j:
        return model.y[i] + height(model, i) <= model.y[j] + M*(1 + model.xij[i,j] - model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_yij = Constraint(model.I, model.J, rule=overlap_yij_rule)

# (xij, yij) = (1, 1): item j is BELOW item i
def overlap_yji_rule(model, i, j):
    if i < j:
        return model.y[j] + height(model, j) <= model.y[i] + M*(2 - model.xij[i,j] - model.yij[i,j]) + M*(2 - model.u[i] - model.u[j])
    return Constraint.Skip
model.overlap_yji = Constraint(model.I, model.J, rule=overlap_yji_rule)

# a placed item must lie inside the W x H floor (a left-out item is released by M*(1 - u[i]))
model.box_xi = Constraint(model.I, rule=lambda model, i: model.x[i] + width(model, i) <= W + M*(1 - model.u[i]))
model.box_yi = Constraint(model.I, rule=lambda model, i: model.y[i] + height(model, i) <= H + M*(1 - model.u[i]))

## Solve

`tee=False` hides the solver log; the cell prints the status (`optimal` means proved best) and the area placed.

In [ ]:
# Solve (the MATLAB original calls intlinprog(problem) with no time limit)
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
print("Status:", results.solver.termination_condition)
print("Objective:", value(model.obj))

## Results

List and draw the placed items inside the dashed 10 × 10 floor. A corner printed as `-0` is 0.

In [ ]:
# Draw the placed items (an item is placed / rotated when its binary is 1; test > 0.5, not truthiness)
placed = [i for i in model.I if model.u[i].value > 0.5]
print("Placed items:", placed, "of", list(model.I))
for i in placed:
    rot = model.r[i].value > 0.5
    print(f"item {i}: {w[i-1]}x{h[i-1]}{' rotated' if rot else ''} at ({model.x[i].value:g}, {model.y[i].value:g})")

fig, ax = plt.subplots()
cmap = plt.get_cmap('Pastel1')
for i in placed:
    wi, hi = (h[i-1], w[i-1]) if model.r[i].value > 0.5 else (w[i-1], h[i-1])
    ax.add_patch(patches.Rectangle((model.x[i].value, model.y[i].value), wi, hi, linewidth=1.5,
                                   edgecolor='black', facecolor=cmap(i / no_items), label=str(i)))
ax.add_patch(patches.Rectangle((0, 0), W, H, fill=False, linestyle='--', edgecolor='grey'))
ax.set_xlim(-1, W + 1); ax.set_ylim(-1, H + 1); ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.grid(True); ax.legend(loc='upper right')
plt.show()

## What do you notice?

The objective is exactly 100, and every optimal layout leaves out item 3. Why that item?

<details><summary>Answer</summary>

To cover 100 of the items' 124 square units, the items left out must have total area exactly $124 - 100 = 24$. The areas are 20, 21, 24, 42, 9 and 8, and the only set of them that adds up to 24 is item 3 on its own. The solver also shows that the other five really fit into the 10 × 10 square, by finding a layout.
</details>

## Try it

Make the floor 10 × 12 (`W, H = 10, 12`) and run all cells again. The items still do not all fit (124 > 120); leaving out only item 6 (area 8) would give 116. What does the solver find?

<details><summary>Answer</summary>

115: items 1, 2, 3, 4 and 6 (20 + 21 + 24 + 42 + 8), leaving out item 5. The five items without item 6 have area 116 ≤ 120 but cannot be arranged in a 10 × 12 rectangle, so area alone is not enough: the shapes must fit too. It solves in about a second.
</details>